In [1]:
import pandas as pd
df = pd.read_csv("C:/Users/AVANTHIKKA/SentinelIQ/data/paysim.csv")

# fraud only happens in these two types
df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
print(df.shape, df["isFraud"].mean())

# features
df["type_cashout"] = (df["type"] == "CASH_OUT").astype(int)
df["hour"] = df["step"] % 24
df["amt_ratio"] = df["amount"] / (df["oldbalanceOrg"] + 1)

# split by time
train = df[df["step"] <= 600]
test = df[df["step"] > 600]
print(len(train), len(test), train["isFraud"].sum(), test["isFraud"].sum())

(2770409, 11) 0.002964544224336551
2726859 43550 6613 1600


In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, average_precision_score

def run(features):
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(class_weight="balanced", max_iter=1000))
    model.fit(train[features], train["isFraud"])
    prob = model.predict_proba(test[features])[:, 1]
    pred = (prob > 0.5).astype(int)
    print("Precision:", round(precision_score(test["isFraud"], pred), 3))
    print("Recall:   ", round(recall_score(test["isFraud"], pred), 3))
    print("PR-AUC:   ", round(average_precision_score(test["isFraud"], prob), 3))